## **Introduction to Integer Programming and Applications with Julia**

<table>
  <tr>
    <td>Chapter</td>
    <td>2 - Knapsack Problems</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

---

## Exercise 2.1

The following Julia code solve the binary, integer models and the relaxations of the knapsack problem giving an instance. The code defines functions to solve both the Binary Knapsack Problem (BKP) and the Integer Knapsack Problem (IKP), as well as their linear relaxations. The results are stored in a DataFrame for easy comparison.


In [1]:
using JuMP       # Modeling language
using HiGHS      # Solver
using DataFrames # Data handling

# Function to solve the Binary Knapsack Problem (BKP) and its linear relaxation
function solve_bkp(v, w, W; relaxation = false)
    model = JuMP.Model(HiGHS.Optimizer)
    JuMP.set_silent(model)
    n = length(v)
    if relaxation
        @variable(model, 0 <= x[1:n] <= 1)
    else
        @variable(model, x[1:n], Bin)
    end
    @objective(model, Max, sum(v[i] * x[i] for i in 1:n))
    @constraint(model, sum(w[i] * x[i] for i in 1:n) <= W)
    JuMP.optimize!(model)
    objective_value = JuMP.objective_value(model)
    solution = JuMP.value.(x)
    weight = sum(w[i] * solution[i] for i in 1:n)
    return objective_value, solution, weight
end

# Function to solve the Integer Knapsack Problem (IKP) and its linear relaxation
function solve_ikp(v, w, W; relaxation = false)
    model = JuMP.Model(HiGHS.Optimizer)
    JuMP.set_silent(model)
    n = length(v)
    if relaxation
        @variable(model, x[1:n] >= 0)
    else
        @variable(model, x[1:n] >= 0, Int)
    end
    @objective(model, Max, sum(v[i] * x[i] for i in 1:n))
    @constraint(model, sum(w[i] * x[i] for i in 1:n) <= W)
    JuMP.optimize!(model)
    objective_value = JuMP.objective_value(model)
    solution = JuMP.value.(x)
    weight = sum(w[i] * solution[i] for i in 1:n)
    return objective_value, solution, weight
end

# Function to solve all four problems and store results in a DataFrame
function solve_knapsack_problems(v, w, W)
    results = DataFrame(
        Problem = ["BKP", "BKP(LP)", "IKP", "IKP(LP)"],
        Objective = [0.0, 0.0, 0.0, 0.0],
        Solution = [Float64[], Float64[], Float64[], Float64[]],
        Weight = [0.0, 0.0, 0.0, 0.0]
    )
    results[1, :Objective], results[1, :Solution], results[1, :Weight] = solve_bkp(v, w, W)
    results[2, :Objective], results[2, :Solution], results[2, :Weight] = solve_bkp(v, w, W; relaxation = true)
    results[3, :Objective], results[3, :Solution], results[3, :Weight] = solve_ikp(v, w, W)
    results[4, :Objective], results[4, :Solution], results[4, :Weight] = solve_ikp(v, w, W; relaxation = true)
    return results
end

solve_knapsack_problems (generic function with 1 method)

### a) Solve the following knapsack problem:

$$
v = (5, 7, 4, 3)\\
𝑤 = (16, 22, 12, 8) \\
𝑊 = 14
$$

In [17]:
# Data
v = [5, 7, 4, 3]    # values
w = [16, 22, 12, 8] # weights
W = 30              # capacity

# Solve the problems and print results
results = solve_knapsack_problems(v, w, W)
println(results)

4×4 DataFrame
 Row │ Problem  Objective  Solution                   Weight  
     │ String   Float64    Array…                     Float64 
─────┼────────────────────────────────────────────────────────
   1 │ BKP        10.0     [0.0, 1.0, -0.0, 1.0]         30.0
   2 │ BKP(LP)    10.1818  [0.0, 0.454545, 1.0, 1.0]     30.0
   3 │ IKP        10.0     [0.0, 0.0, 1.0, 2.0]          28.0
   4 │ IKP(LP)    11.25    [0.0, 0.0, 0.0, 3.75]         30.0


#### Discussion

In BKP, only one copy of each item is allowed. The optimal solution selects items 2 and 4, exactly filling the knapsack to capacity (30 units) and achieving an objective value of 10.

In BKP(LP), you can select fractional parts of a item. Here, item 2 is partially selected (about 0.45), and items 3 and 4 are fully selected, slightly increasing the objective value to 10.18. The knapsack is fully utilized. The gap is $z_{BKP(LP)} - z_{BKP} = $ 10.18 - 10 = 0.18.

In IKP, multiple copies of each item are allowed, but the optimal solution selects one of item 3 and two of item 4, reaching a total value of 10 and a weight of 28 (not fully utilizing the capacity due to integer constraints).

In IKP(LP), both repetition and fractional quantities are allowed. The solution allocates all capacity to item 4 (3.75 units), achieving the highest objective value (11.25) and fully utilizing the knapsack. The gap is $z_{IKP(LP)} - z_{IKP} = $ 11.25 - 10 = 1.25.

The results illustrate three important optimization concepts:

a) Relaxation increases the objective value (for maximization) enlarging the feasible region, so the objective value can only improve (or remain equal)

b) The hierarchy of objective values reflects the progressive enlargement of the feasible region: binary constraints are the most restrictive, LP relaxation removes integrality, integer multiplicities allow repeated items, and full relaxation allows both repetition and fractional quantities.

c) Both LP solutions tend to exploit the best density items (4 and 3)

$$
\left(\frac{v_1}{w_1},\frac{v_2}{w_2},\frac{v_3}{w_3},\frac{v_4}{w_4}\right)
=
\left(\frac{5}{16}, \frac{7}{22},\frac{4}{12},\frac{3}{8}\right)
=
(
\overset{4}{0.31},
\overset{3}{0.32},
\overset{2}{0.33},
\overset{1}{0.38}
)
$$

These results demonstrate how modeling assumptions strongly influence both the structure of the optimal solution and the achievable profit.

---

## b) Solve the following knapsack problem:

$$
v = (4, 2, 10, 2, 1)\\
𝑤 = (12, 2, 4, 1, 1) \\
𝑊 = 15
$$

In [21]:
# Data
v = [4, 2, 10, 2, 1] # values
w = [12, 2, 4, 1, 1] # weights
W = 15               # capacity

# Solve the problems and print results
results = solve_knapsack_problems(v, w, W)
println(results)

4×4 DataFrame
 Row │ Problem  Objective  Solution                        Weight  
     │ String   Float64    Array…                          Float64 
─────┼─────────────────────────────────────────────────────────────
   1 │ BKP        15.0     [0.0, 1.0, 1.0, 1.0, 1.0]           8.0
   2 │ BKP(LP)    17.3333  [0.583333, 1.0, 1.0, 1.0, 1.0]     15.0
   3 │ IKP        36.0     [0.0, 0.0, 3.0, 3.0, 0.0]          15.0
   4 │ IKP(LP)    37.5     [0.0, 0.0, 3.75, 0.0, 0.0]         15.0


In [20]:
v ./ w

5-element Vector{Float64}:
 0.3333333333333333
 1.0
 2.5
 2.0
 1.0

#### Discussion

In BKP a notable observation is that the knapsack is not completely filled, since only 8 of the 15 capacity units are used. This happens because the remaining item (item 1) is too heavy to fit in the remaining space. The binary restriction significantly limits the feasible region and leads to the smallest objective value among all models. In the relaxation BKP(LP), there is an increase in the objective value and the knapsack is fully filled. The improvement occurs because the LP relaxation enlarges the feasible region by allowing fractional decisions. Therefore, the LP relaxation provides an upper bound for the binary problem ($z_{BKP(LP)} \ge z_{BKP}$), and the gap is $17.33 - 15 = 2.33$.

In the integer version (IKP) multiple copies of each item are allowed. The objective value rises dramatically to 36 because item 3 has the best value-to-weight ratio (10/4 = 2.5) and item 4 efficiently fills the remaining capacity. Compared to BKP, the feasible region is much larger because items can be repeated. This explains the large increase in profit. Thus, allowing repetitions fundamentally changes the structure of the optimal solution. Finally, in the relaxed integer model IKP(LP), the optimal strategy is to allocate all capacity to the item with the highest value-to-weight ratio. Since item 3 dominates (10/4 = 2.5) the solution selects 3.75 of this item. This is the largest objective value among all models because the feasible region is the largest possible. Again, the LP relaxation provides an upper bound ($z_{IKP(LP)} \ge z_{IKP}$) and the gap is relatively small $z_{IKP(LP)} - z_{IKP} = 37.5 - 36 = 1.5$.

---

### c) Solve the following knapsack problem:

$$
v = (3, 9, 6, 3, 2)\\
𝑤 = (8, 20, 12, 6, 4) \\
𝑊 = 17
$$

In [24]:
# Data
v = [3, 9, 6, 3, 2]  # values
w = [8, 5, 12, 6, 4] # weights
W = 17               # capacity

# Solve the problems and print results
results = solve_knapsack_problems(v, w, W)
println(results)

4×4 DataFrame
 Row │ Problem  Objective  Solution                        Weight  
     │ String   Float64    Array…                          Float64 
─────┼─────────────────────────────────────────────────────────────
   1 │ BKP           15.0  [0.0, 1.0, 1.0, 0.0, 0.0]          17.0
   2 │ BKP(LP)       15.0  [0.0, 1.0, 0.166667, 1.0, 1.0]     17.0
   3 │ IKP           27.0  [0.0, 3.0, 0.0, 0.0, -0.0]         15.0
   4 │ IKP(LP)       30.6  [0.0, 3.4, 0.0, 0.0, 0.0]          17.0


#### Discussion

In BKP, only one copy of each item is allowed. The optimal solution selects items 2 and 3, reaching a total value of 15 and a weight of 17 (fully utilizing the capacity). In BKP(LP), you can select fractional parts of a item. Here, item 3 is partially selected (about 0.16), and items 2, 4 and 5 are fully selected, acheiving the same objective value to 15 and fully utilizing the knapsack. The gap is $z_{BKP(LP)} - z_{BKP} = 17 - 17 = 0$. The LP solution is fractional, yet it achieves exactly the same objective value as the integer optimum. This means there are multiple optimal LP solutions, one of which happens to be fractional.

In IKP, multiple copies of each item are allowed, but the optimal solution is composed only by items of type 2 since they have the highest value-to-weight ratio $\left(\frac{v_2}{w_2} = 1.8\right)$ and the solution is limited by the weight of this item. In IKP(LP), both repetition and fractional quantities are allowed. The solution allocates all capacity to item 3 (about 3.4 units), achieving the highest objective value (30.6) and fully utilizing the knapsack. The gap is $z_{IKP(LP)} - z_{IKP} = 17 - 15 = 2$.